# UC07 — Predicción de Demanda Hídrica**Objetivo:** Predecir la demanda de agua a 24-72 horas combinando patrones de consumo, meteorología y calendario.**Tecnologías:** SNOWFLAKE.ML.FORECAST · Streamlit**Tiempo estimado:** 13 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS DEMANDA_HIDRICA_DB;USE DATABASE DEMANDA_HIDRICA_DB;USE SCHEMA PUBLIC;CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Series de Consumo Diario2 años (730 días) de demanda diaria con estacionalidad, efecto fin de semana y festivos.

In [ ]:
CREATE OR REPLACE TABLE consumo_diario ASSELECT    DATEADD('day', -SEQ4(), CURRENT_DATE()) AS fecha,    ROUND(80000 + 20000 * SIN(SEQ4() * 3.14159 / 182.5) + CASE WHEN DAYOFWEEK(DATEADD('day', -SEQ4(), CURRENT_DATE())) IN (0,6) THEN -12000 ELSE 0 END + UNIFORM(-8000, 8000, RANDOM()), 0) AS demanda_m3,    ROUND(15 + 12 * SIN(SEQ4() * 3.14159 / 182.5) + UNIFORM(-3, 3, RANDOM()), 1) AS temp_max_c,    ROUND(GREATEST(0, CASE WHEN UNIFORM(0,100,RANDOM()) < 20 THEN UNIFORM(1, 40, RANDOM()) ELSE 0 END), 1) AS precipitacion_mm,    CASE WHEN UNIFORM(0, 100, RANDOM()) < 3 THEN TRUE ELSE FALSE END AS es_festivoFROM TABLE(GENERATOR(ROWCOUNT => 730));SELECT COUNT(*) AS dias, ROUND(AVG(demanda_m3),0) AS demanda_media, MIN(demanda_m3) AS min_demanda, MAX(demanda_m3) AS max_demanda FROM consumo_diario;

## Paso 3 — Entrenar Modelo de Forecasting

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST predictor_demanda(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'consumo_diario'),    TIMESTAMP_COLNAME => 'fecha',    TARGET_COLNAME => 'demanda_m3');

## Paso 4 — Generar Predicciones a 72 Horas

In [ ]:
CALL predictor_demanda!FORECAST(FORECASTING_PERIODS => 7);CREATE OR REPLACE TABLE predicciones_demanda ASSELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));SELECT * FROM predicciones_demanda ORDER BY TS;

## Paso 5 — Planificación de ProducciónTraducimos demanda prevista a parámetros operativos de planta.

In [ ]:
CREATE OR REPLACE VIEW plan_produccion ASSELECT    TS AS fecha,    ROUND(FORECAST, 0) AS demanda_predicha_m3,    CEIL(FORECAST / 24 / 300) AS lineas_tratamiento_necesarias,    ROUND(FORECAST / 24, 0) AS caudal_produccion_m3h,    ROUND(FORECAST * 1.1, 0) AS produccion_con_margen_m3FROM predicciones_demanda;SELECT * FROM plan_produccion;

## Paso 6 — Dashboard de Planificación

In [ ]:
import streamlit as stfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()st.title("📊 Predicción de Demanda Hídrica")pred = session.sql("SELECT * FROM predicciones_demanda ORDER BY TS").to_pandas()col1, col2 = st.columns(2)col1.metric("Demanda Mañana", f"{pred['FORECAST'].iloc[0]:,.0f} m³")col2.metric("Demanda 7 días", f"{pred['FORECAST'].sum():,.0f} m³")st.line_chart(pred.set_index("TS")[["FORECAST", "LOWER_BOUND", "UPPER_BOUND"]])st.subheader("Plan de Producción")plan = session.sql("SELECT * FROM plan_produccion").to_pandas()st.dataframe(plan, use_container_width=True)

## Paso 7 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS DEMANDA_HIDRICA_DB;